# 2 - Long-term memory: semantic, episodic, procedural

Three independent stores, split by **access pattern**:

| Store | What goes in | Where | When written |
|---|---|---|---|
| **Semantic** | natural-language memories ("user loves dogs") | Chroma (vector) | agent calls `semantic_write` |
| **Episodic** | thread summaries | Chroma (vector) | end of thread, via reflection |
| **Procedural** | reusable prompt fragments ("skills") | SQLite | end of thread, via reflection |

- Semantic answers *"what do I durably know about this user/world?"* — agent-curated, retrieved by similarity.
- Episodic answers *"have I done a thread like this before?"*
- Procedural answers *"what prompt fragments should I always carry?"*

**Why tool-driven semantic writes?** A prior version auto-extracted (subject, predicate, object) triples from every turn, polluting memory with conversational noise. The new shape: `semantic_write(text)` and `semantic_search(query)`, called only when the agent decides the memory is worth keeping.

Scenario simulates Segment 6's SDR: three back-to-back conversations with prospects from different companies.

In [1]:
import _path_setup  # noqa: F401

import os
import shutil
import uuid
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENROUTER_API_KEY"), "set OPENROUTER_API_KEY in .env"

from memory import (
    SemanticMemory,
    EpisodicMemory,
    ProceduralMemory,
    reflect_on_thread,
)
from shared import get_llm

# Wipe stores so each notebook run is a clean slate.
MEM = Path("data/memory_demo")
if MEM.exists():
    shutil.rmtree(MEM)
MEM.mkdir(parents=True)

semantic = SemanticMemory(MEM / "semantic_chroma")
episodic = EpisodicMemory(MEM / "episodic_chroma")
procedural = ProceduralMemory(MEM / "procedural.sqlite")

MODEL = "openai/gpt-5.4-nano"
llm = get_llm(MODEL)  # used by the memory-augmented + naive comparison cells later
print("memory stores initialized at", MEM)

/Users/sinanozdemir/Teaching/Pearson/advanced-agentic-ai-in-three-weeks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8506.71it/s]


memory stores initialized at data/memory_demo


## The scenario: three prospect conversations

Tiny scripted exchanges (user = prospect, agent responds). Scripted, not LLM-generated, so per-turn facts are predictable and the test is honest.

(In Segment 6 these come from real HubSpot contacts via the SDR app; inline here for self-containment.)

In [2]:
CONVERSATIONS = [
    {
        "thread_id": "thread-acme",
        "turns": [
            ("Hi, I'm Anna from Acme Corp. We're a 250-person fintech in Berlin, and I'm the VP of Engineering. Tell me how your tool helps teams like ours.",),
            ("We're currently using LangChain in production but the observability has been painful. We process about 4M agent runs a month.",),
            ("Pricing — what would 4M runs/month cost on your platform?",),
        ],
    },
    {
        "thread_id": "thread-globex",
        "turns": [
            ("This is Brian, CTO at Globex. We're an 80-person healthcare SaaS in Boston. We're starting to evaluate agent observability tools.",),
            ("Our compliance team needs SOC2 and HIPAA. Do you cover those?",),
            ("How do customers like Acme typically roll this out?",),
        ],
    },
    {
        "thread_id": "thread-initech",
        "turns": [
            ("Hi, Carol from Initech here. We're a small AI startup, 12 people, building dev tools. Looking for cheap observability — Langfuse free tier wasn't enough.",),
            ("We're paying $0/month right now. What's your starter plan?",),
        ],
    },
]

SYSTEM_PROMPT = (
    "You are an SDR for an LLM agent observability platform. Be concise (max 3 sentences). "
    "Make pricing claims if needed for realism. The teaching point isn't sales quality — "
    "it's that the FACTS the prospect shares (company, size, stack, requirements) end up "
    "in semantic memory."
)

print(f"{len(CONVERSATIONS)} threads, {sum(len(c['turns']) for c in CONVERSATIONS)} turns total")

3 threads, 8 turns total


## Run the conversations — semantic memory as an explicit tool

The agent decides when to persist a memory. Bind `semantic_write`, instruct the model to log durable facts, let it call the tool zero or more times per turn.

After each thread ends, `reflect_on_thread(...)` does the episodic + procedural writes.

In [3]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool

# Closure state so each tool call is attributed to the right thread.
_ACTIVE = {"thread_id": ""}


@tool
def semantic_write(text: str) -> str:
    """Store a single durable, natural-language memory about the user or their company
    (e.g. 'Acme Corp is a 250-person fintech in Berlin'). Call once per durable fact;
    skip transient or conversational content."""
    rid = semantic.write(text.strip(), thread_id=_ACTIVE["thread_id"])
    return f"stored {rid}"


@tool
def semantic_search(query: str, k: int = 5) -> str:
    """Recall durable memories similar to `query` from semantic memory."""
    hits = semantic.search(query, k=int(k))
    if not hits:
        return "no matching memories"
    return "\n".join(f"- [{h.score:.2f}] {h.text}" for h in hits)


MEM_SYSTEM = (
    SYSTEM_PROMPT
    + "\n\nYou have two memory tools:\n"
    "  - semantic_write(text): store a short natural-language memory about the prospect\n"
    "    (role, company, size, location, stack, compliance, pricing constraints).\n"
    "    Call it once per durable fact. Skip greetings and your own pitch.\n"
    "  - semantic_search(query): recall what you already know before answering.\n"
    "Be intentional — only write things worth carrying into a future thread."
)

agent = create_agent(
    model=get_llm(MODEL),
    tools=[semantic_write, semantic_search],
    system_prompt=MEM_SYSTEM,
)

transcripts = {}

for convo in CONVERSATIONS:
    tid = convo["thread_id"]
    print(f"\n=== {tid} ===")
    messages: list = []
    transcript: list[dict] = []
    for turn_idx, (user_text,) in enumerate(convo["turns"]):
        _ACTIVE["thread_id"] = tid
        messages.append(HumanMessage(content=user_text))

        out = agent.invoke({"messages": messages})
        messages = out["messages"]

        n_writes = sum(
            1
            for m in messages
            if isinstance(m, AIMessage)
            for tc in (getattr(m, "tool_calls", None) or [])
            if (tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", "")) == "semantic_write"
        )

        final = next(
            (m for m in reversed(messages)
             if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None)),
            None,
        )
        assistant_text = (final.content if final is not None else "") or ""
        if not isinstance(assistant_text, str):
            assistant_text = str(assistant_text)

        transcript.append({"role": "user", "content": user_text})
        transcript.append({"role": "assistant", "content": assistant_text})
        print(f"  turn {turn_idx}: agent wrote {n_writes} memories cumulatively")

    transcripts[tid] = transcript
    # End-of-thread reflection: one LLM call that (1) writes a summary into
    # episodic memory and (2) proposes 0-3 new procedural skills if the
    # agent did something noteworthy. Semantic memory is NOT touched here —
    # it was already written turn-by-turn via the agent's `semantic_write`
    # tool. This split keeps the hot path cheap and pushes expensive
    # cross-thread reasoning into a once-per-thread batch.
    # rubric_score is the "should I trust what I learned here" signal.
    # Stored on the episodic entry (so retrieval can prefer threads that
    # went well) and on every procedural skill proposed (procedural.py
    # ranks `ORDER BY score DESC` when injecting skills into future system
    # prompts). Hardcoded to 4.0 here because these toy convos don't have
    # a real rubric judge — in the SDR app / Forge this is the actual
    # judge's 0-5 score for the thread.
    refl = reflect_on_thread(
        thread_id=tid, messages=transcript,
        episodic=episodic, procedural=procedural,
        rubric_score=4.0, model_slug=MODEL,
    )
    print(f"  reflection: skills={refl['skills']}")

print("\nDone.")
print(f"semantic memories: {semantic.count()}")
print(f"episodic entries:  {episodic.count()}")
print(f"procedural skills: {procedural.count()}")


=== thread-acme ===
  turn 0: agent wrote 1 memories cumulatively
  turn 1: agent wrote 2 memories cumulatively
  turn 2: agent wrote 2 memories cumulatively
  reflection: skills=['ask_pricing_dependencies_upfront']

=== thread-globex ===
  turn 0: agent wrote 1 memories cumulatively
  turn 1: agent wrote 2 memories cumulatively
  turn 2: agent wrote 2 memories cumulatively
  reflection: skills=[]

=== thread-initech ===
  turn 0: agent wrote 1 memories cumulatively
  turn 1: agent wrote 1 memories cumulatively
  reflection: skills=[]

Done.
semantic memories: 5
episodic entries:  3
procedural skills: 1


## Inspect semantic memory: what did the agent decide to remember?

Agent-driven writes: fewer rows than triple extraction, each row a complete sentence the model thought worth keeping.

In [4]:
import pandas as pd
from dataclasses import asdict

records = semantic.all()
df_mem = pd.DataFrame([asdict(r) for r in records])
print(f"Total memories: {len(df_mem)}\n")
df_mem[["id", "text", "thread_id", "created_at"]]

Total memories: 5



,id,text,thread_id,created_at
0,sm-2318c0f4a43d,"Anna is VP of Engineering at Acme Corp, a 250-...",thread-acme,2026-05-06T21:23:43.694075+00:00
1,sm-54e4fc9ff35b,Acme Corp processes about 4M agent runs per mo...,thread-acme,2026-05-06T21:23:49.475264+00:00
2,sm-65ef5bccade5,Brian is CTO at Globex; Globex is an 80-person...,thread-globex,2026-05-06T21:23:55.913375+00:00
3,sm-a05c67119d20,Globex needs agent observability tooling that ...,thread-globex,2026-05-06T21:23:57.987217+00:00
4,sm-4c81ebe16ba5,Carol from Initech: small AI startup (12 peopl...,thread-initech,2026-05-06T21:24:03.885130+00:00


## Inspect episodic memory: can we retrieve the right past thread?

When a *new* prospect resembles an old one, does similarity search return the right thread?

In [5]:
import textwrap
for cue in [
    "prospect is a fintech in Europe asking about pricing",
    "healthcare company needs SOC2 / HIPAA compliance",
    "tiny AI startup that needs a free tier",
]:
    print(f"\nCUE: {cue}")
    for hit in episodic.search(cue, k=2):
        print(f"  -> [{hit.thread_id}] score={hit.score:.2f}")
        print(textwrap.indent(textwrap.fill(hit.summary, 90), "     "))


CUE: prospect is a fintech in Europe asking about pricing
  -> [thread-acme] score=4.00
     Anna (Acme Corp) asked how the agent observability tool helps fintech teams like hers,
     then stated they use LangChain and experience painful observability at ~4M agent
     runs/month, and finally requested pricing for that volume. The agent explained the value
     at scale, offered to map integrations to her stack, and gave only a rough pricing band
     while requesting retention and feature details to produce a concrete estimate. The
     conversation was partially successful: it addressed the pricing question directionally but
     did not provide an exact cost without the follow-up inputs.
  -> [thread-initech] score=4.00
     The user (Carol from Initech) asked for a cheap observability solution for their 12-person
     AI dev tools startup, noting that Langfuse’s free tier was insufficient. The agent
     acknowledged the constraints and proposed narrowing requirements (deployment

## Inspect procedural memory: what skills did the agent decide to remember?

Reflection is conservative on purpose — most threads end with `skills=[]`. We expect 0–3 skills total across the three threads.

In [6]:
for s in procedural.all():
    print(f"=== {s.name}  (used {s.usage_count}x, score {s.score})")
    if s.when_to_use:
        print(f"   when: {s.when_to_use}")
    print(f"   {s.fragment}\n")

print("--- system-prompt rendering ---")
print(procedural.render_for_system_prompt(n=5) or "(no skills extracted yet)")

=== ask_pricing_dependencies_upfront  (used 1x, score 4.0)
   when: User asks 'what would X runs/month cost' without specifying retention or storage/replay needs.
   When asked for pricing at a specific run volume, quickly request the key variables that drive cost (e.g., trace retention window, whether full payload/tool traces are stored vs metrics-only, and which features like replay/artifact storage/alerting are enabled). Provide a provisional range only if those inputs are missing, and clearly state what data you need to tighten the number.

--- system-prompt rendering ---
# Learned skills (procedural memory)

## ask_pricing_dependencies_upfront
_Use when: User asks 'what would X runs/month cost' without specifying retention or storage/replay needs._
When asked for pricing at a specific run volume, quickly request the key variables that drive cost (e.g., trace retention window, whether full payload/tool traces are stored vs metrics-only, and which features like replay/artifact stora

## Use the memory: a 4th conversation with a similar prospect

New prospect resembles Acme. Before responding: (a) pull semantic facts about the company, (b) pull the most similar episodic thread, (c) inject relevant procedural skills. Watch the response personalize.

In [7]:
new_prospect_msg = (
    "Hi, I'm Diana from Acme Corp — taking over from Anna. We're still considering observability tools. "
    "Where did our last conversation leave off, and what did you recommend?"
)

# 1. Recall what we already know — embedding similarity, not exact lookup
known = semantic.search("Acme Corp", k=5, min_score=0.25)
fact_block = "\n".join(f"- {h.text}" for h in known) or "(none)"

# 2. Find similar past threads
past = episodic.search(new_prospect_msg, k=1)
past_block = past[0].summary if past else "(no similar thread)"

# 3. Inject any relevant procedural skills
skills_block = procedural.render_for_system_prompt(n=3)

memory_aware_system_prompt = (
    SYSTEM_PROMPT + "\n\n"
    "# What we remember about this prospect's company\n" + fact_block + "\n\n"
    "# Most similar past thread\n" + past_block + "\n\n"
    + skills_block
)

print("--- memory-augmented system prompt ---")
print(memory_aware_system_prompt)
print("\n--- agent response ---")
resp = llm.invoke([SystemMessage(content=memory_aware_system_prompt),
                   HumanMessage(content=new_prospect_msg)])
print(resp.content if isinstance(resp.content, str) else str(resp.content))

--- memory-augmented system prompt ---
You are an SDR for an LLM agent observability platform. Be concise (max 3 sentences). Make pricing claims if needed for realism. The teaching point isn't sales quality — it's that the FACTS the prospect shares (company, size, stack, requirements) end up in semantic memory.

# What we remember about this prospect's company
- Acme Corp processes about 4M agent runs per month and uses LangChain in production; observability is currently painful.
- Anna is VP of Engineering at Acme Corp, a 250-person fintech in Berlin.
- Brian is CTO at Globex; Globex is an 80-person healthcare SaaS company in Boston evaluating agent observability tools.

# Most similar past thread
Anna (Acme Corp) asked how the agent observability tool helps fintech teams like hers, then stated they use LangChain and experience painful observability at ~4M agent runs/month, and finally requested pricing for that volume. The agent explained the value at scale, offered to map integratio

## Compare: same prompt, no memory

In [8]:
naive_resp = llm.invoke([SystemMessage(content=SYSTEM_PROMPT),
                         HumanMessage(content=new_prospect_msg)])
print("--- response WITHOUT memory ---\n")
print(naive_resp.content if isinstance(naive_resp.content, str) else str(naive_resp.content))

--- response WITHOUT memory ---

Hi Diana—last time you shared that Acme is evaluating observability tooling, but I don’t have access to any prior chat history with Anna in this thread. If you tell me Acme’s rough size, current stack (cloud, language/runtime, Kubernetes?), and the top requirements (logs vs metrics vs traces, APM, alerting, cost/retention), I’ll restate what we recommended and propose a short shortlist.


## Recap

1. **Vector vs SQLite by access pattern, not dogma.** Semantic + episodic in Chroma (similarity retrieval); procedural in SQLite (tiny boot read).
2. **Tool-driven writes change the failure mode.** Auto-extraction failed at "garbage in memory"; tool writes fail at "agent forgot to write" — a prompt+eval problem, easier to debug.
3. **Memory-augmented responses personalize.** The agent receives recalled memories in its system prompt and can `semantic_search` mid-turn when it needs more.
